In [ ]:
%pip install pyarrow

In [30]:
%pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   - -------------------------------------- 3.1/101.7 MB 18.5 MB/s eta 0:00:06
   -- ------------------------------------- 7.1/101.7 MB 19.0 MB/s eta 0:00:05
   ---- ----------------------------------- 12.1/101.7 MB 21.0 MB/s eta 0:00:05
   ------ --------------------------------- 16.5/101.7 MB 20.8 MB/s eta 0:00:05
   -------- ------------------------------- 20.7/101.7 MB 20.4 MB/s eta 0:00:04
   --------- ------------------------------ 24.9/101.7 MB 20.5 MB/s eta 0:00:04
   ----------- ---------------------------- 29.1/101.7 MB 20.5 MB/s eta 0:00:04
   ------------- -------------------------- 33.3/101.7 MB 20.3 MB/s eta 0:00:04
   -------------- ------------------------- 37.7/101.7 MB 20.3 MB/s eta 0:00:04
   ---------------- ----------------------- 42.5/101.7 MB 20.6 MB/s eta 0:00:03
   ------------------ --------------------- 46.7/101.7 MB 20.5 MB/s eta 0:00:03
   -------------------- ------------------- 51.1/10

In [33]:
import numpy as np

In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# On charge le fichier de données
df = pd.read_parquet(r'C:\Users\manug\OneDrive\Desktop\ML_2026\train.parquet')

# Afficher les 5 premières lignes pour découvrir les noms exacts des colonnes
display(df.head())


In [18]:
# Remplace par ton vrai chemin absolu, comme tu l'as fait pour train.parquet
chemin_sensors = r"C:\Users\manug\OneDrive\Desktop\ML_2026\sensors.parquet"

df_sensors = pd.read_parquet(chemin_sensors)
display(df_sensors.head())

,sensor,coor_x,coor_y,coor_z
0,N2,0.5,0.0,0.0
1,N4,1.4,0.0,0.0
2,N5,0.5,2.4,0.0
3,N6,0.0,2.4,0.0
4,N7,0.0,3.5,0.0


In [19]:
# On fusionne les données d'entraînement avec les positions des capteurs
# 'on="sensor"' indique la colonne commune qui sert de clé
# 'how="left"' s'assure qu'on garde bien toutes nos lignes de temps
df_complet = pd.merge(df, df_sensors, on='sensor', how='left')

# On vérifie le résultat
display(df_complet.head())

,sensor,time,power,temperature,coor_x,coor_y,coor_z
0,N102,0.0,1487.964722,17.514429,46.131474,3.5,0.0
1,N102,864000.0,1487.288818,17.820795,46.131474,3.5,0.0
2,N102,1728000.0,1486.612915,17.573187,46.131474,3.5,0.0
3,N102,2592000.0,1485.936890,16.513235,46.131474,3.5,0.0
4,N102,3456000.0,1485.260986,16.303427,46.131474,3.5,0.0


In [ ]:
# Affiche la liste des capteurs et combien de fois ils apparaissent
df['sensor'].value_counts()

In [ ]:
# On isole uniquement les données du capteur N102
capteur_N102 = df[df['sensor'] == 'N102']

# On crée le graphique pour la puissance
plt.figure(figsize=(10, 5))
plt.plot(capteur_N102['time'], capteur_N102['power'], marker='.', linestyle='-')
plt.title('Évolution de la puissance pour le capteur N102')
plt.xlabel('Temps')
plt.ylabel('Puissance')
plt.grid(True)
plt.show()

In [ ]:
# On garde les données triées du capteur N102
capteur_N102 = df[df['sensor'] == 'N102'].sort_values(by='time')

# Création d'un graphique avec deux axes Y (pour avoir des échelles différentes)
fig, ax1 = plt.subplots(figsize=(12, 6))

color = 'tab:blue'
ax1.set_xlabel('Temps (secondes)')
ax1.set_ylabel('Puissance', color=color)
ax1.plot(capteur_N102['time'], capteur_N102['power'], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # On crée un deuxième axe Y qui partage le même axe X
color = 'tab:red'
ax2.set_ylabel('Température (°C)', color=color)
ax2.plot(capteur_N102['time'], capteur_N102['temperature'], color=color, alpha=0.7)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Évolution de la puissance et de la température (Capteur N102)')
fig.tight_layout()
plt.show()

In [20]:
# Importation de l'outil scikit-learn pour séparer les données
from sklearn.model_selection import train_test_split

# 1. On isole la CIBLE (ce qu'on veut prédire : y)
y = df_complet['temperature']

# 2. On isole les CARACTÉRISTIQUES (les indices pour deviner : X)
# On garde le temps, la puissance, et les coordonnées 3D.
# On exclut le nom du capteur ("N102") car un algorithme mathématique ne sait lire que des nombres.
X = df_complet[['time', 'power', 'coor_x', 'coor_y', 'coor_z']]

# 3. On coupe le jeu de données en deux (80% pour apprendre, 20% pour l'examen)
# random_state=42 permet de faire une coupe aléatoire, mais qui sera la même à chaque fois que tu relances le code.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# On affiche le résultat pour vérifier
print(f"L'IA va s'entraîner sur {X_train.shape[0]} lignes.")
print(f"L'IA sera testée sur {X_test.shape[0]} lignes.")

L'IA va s'entraîner sur 5301542 lignes.
L'IA sera testée sur 1325386 lignes.


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. On initialise notre nouveau modèle (plus intelligent)
# n_estimators=50 veut dire qu'on crée 50 arbres de décision. 
# n_jobs=-1 permet d'utiliser toute la puissance de ton processeur pour aller plus vite.
modele_rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)

# 2. L'entraînement
print("Entraînement de la Forêt Aléatoire en cours (patiente un peu, ça peut prendre 1 à 2 minutes)...")
modele_rf.fit(X_train, y_train)
print("Entraînement terminé ! 🌲🎉\n")

# 3. L'examen et la correction
predictions_rf = modele_rf.predict(X_test)
erreur_moyenne_rf = mean_absolute_error(y_test, predictions_rf)
score_r2_rf = r2_score(y_test, predictions_rf)

print("--- NOUVEAUX RÉSULTATS ---")
print(f"Erreur moyenne : L'IA se trompe de {erreur_moyenne_rf:.2f} °C en moyenne.")
print(f"Score R²       : {score_r2_rf:.2f}")

Entraînement de la Forêt Aléatoire en cours (patiente un peu, ça peut prendre 1 à 2 minutes)...
Entraînement terminé ! 🌲🎉

--- NOUVEAUX RÉSULTATS ---
Erreur moyenne : L'IA se trompe de 1.43 °C en moyenne.
Score R²       : 1.00


In [24]:
# 1. Charger le fichier d'examen (test)
# N'oublie pas de mettre le vrai chemin de ton fichier
chemin_test = r"C:\Users\manug\OneDrive\Desktop\ML_2026\test.parquet" 
df_test = pd.read_parquet(chemin_test)

# 2. On fusionne avec les capteurs pour donner les coordonnées X, Y, Z à l'IA
df_test_complet = pd.merge(df_test, df_sensors, on='sensor', how='left')

# 3. On donne à l'IA exactement les mêmes colonnes qu'elle a apprises
X_examen = df_test_complet[['time', 'power', 'coor_x', 'coor_y', 'coor_z']]

# 4. L'IA passe l'examen et devine les températures !
predictions_finales = modele_rf.predict(X_examen)

# 5. On affiche les premières lignes de test pour voir à quoi elles ressemblent
display(df_test.head())

print("\nEt voici les 5 premières températures devinées par ton IA :")
print(predictions_finales[:5])

,sensor,time,power
0,N104,864000.0,1487.288818
1,N104,1728000.0,1486.612915
2,N104,2592000.0,1485.936890
3,N104,3456000.0,1485.260986
4,N104,4320000.0,1484.585083



Et voici les 5 premières températures devinées par ton IA :
[17.40608366 17.10498886 17.43039886 17.54967163 17.37516045]


In [27]:
# 2. On crée le tableau avec la bonne majuscule attendue par Kaggle !
soumission = pd.DataFrame({
    'Id': X_examen.index,  # <-- C'est ici le changement : un grand 'I'
    'temperature': predictions_finales
})

# 3. Sauvegarde en CSV (ça va écraser l'ancien fichier avec la version corrigée)
chemin_csv = 'ma_premiere_soumission.csv'
soumission.to_csv(chemin_csv, index=False)

print(f"\nFichier corrigé et sauvegardé avec succès ! 🚀")
display(soumission.head())


Fichier corrigé et sauvegardé avec succès ! 🚀


,Id,temperature
0,0,17.406084
1,1,17.104989
2,2,17.430399
3,3,17.549672
4,4,17.375160


In [28]:
# 1. CORRECTION : On supprime les doublons de capteurs (on garde juste la 1ère position trouvée)
df_sensors_unique = df_sensors.drop_duplicates(subset=['sensor'], keep='first')

# 2. On refait la fusion proprement avec ce tableau nettoyé
df_test_complet = pd.merge(df_test, df_sensors_unique, on='sensor', how='left')

# 3. On extrait les caractéristiques pour l'IA
X_examen = df_test_complet[['time', 'power', 'coor_x', 'coor_y', 'coor_z']]

# 4. On relance les prédictions
predictions_finales = modele_rf.predict(X_examen)

# 5. On emballe le tout dans le fichier de soumission
soumission = pd.DataFrame({
    'Id': df_test.index, 
    'temperature': predictions_finales
})

# 6. Sauvegarde
chemin_csv = 'ma_premiere_soumission.csv'
soumission.to_csv(chemin_csv, index=False)

# Vérification finale pour se rassurer avant d'envoyer
print(f"Lignes attendues par Kaggle : {len(df_test)}")
print(f"Lignes dans ton fichier CSV : {len(soumission)}")
if len(df_test) == len(soumission):
    print("✅ Le nombre de lignes correspond parfaitement. Tu peux uploader sur Kaggle ! 🚀")

Lignes attendues par Kaggle : 2190480
Lignes dans ton fichier CSV : 2190480
✅ Le nombre de lignes correspond parfaitement. Tu peux uploader sur Kaggle ! 🚀


In [31]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

# 1. FEATURE ENGINEERING : On crée une variable 'distance'
# La physique nous dit que la distance à la source compte !
df_propre['dist_origin'] = (df_propre['coor_x']**2 + df_propre['coor_y']**2 + df_propre['coor_z']**2)**0.5

# On met à jour nos X
features = ['time', 'power', 'coor_x', 'coor_y', 'coor_z', 'dist_origin']
X = df_propre[features]
y = df_propre['temperature']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. XGBOOST : Le tueur de compétitions
# On ajuste les 'hyperparamètres' pour être ultra précis
modele_ultra = xgb.XGBRegressor(
    n_estimators=1000,      # On laisse l'IA apprendre plus longtemps
    learning_rate=0.05,     # On avance à petits pas pour ne rien rater
    max_depth=8,            # Des arbres plus profonds pour plus de complexité
    tree_method='hist',     # Pour que ça aille super vite
    device="cuda"           # Si tu as une carte graphique NVIDIA, sinon enlève cette ligne
)

print("Entraînement de l'artillerie lourde...")
modele_ultra.fit(X_train, y_train)

# 3. ÉVALUATION
preds = modele_ultra.predict(X_test)
print(f"Nouvelle erreur MAE : {mean_absolute_error(y_test, preds):.4f}")

Entraînement de l'artillerie lourde...


c:\Users\manug\anaconda3\envs\introml\Lib\site-packages\xgboost\core.py:751: UserWarning: [16:11:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Nouvelle erreur MAE : 0.9838


In [36]:
import numpy as np
import pandas as pd

# 1. On s'assure que le modèle est bien celui qu'on vient d'entraîner
# Si tu as appelé ton modèle 'modele_ultra' plus haut, remplace le nom ici :
mon_modele = modele_final 

# 2. Préparation du fichier test
df_test_final = pd.merge(df_test, df_sensors_unique, on='sensor', how='left')

# CALCUL DE LA DISTANCE (indispensable pour l'IA)
df_test_final['dist_origin'] = np.sqrt(df_test_final['coor_x']**2 + df_test_final['coor_y']**2 + df_test_final['coor_z']**2)

# 3. Sélection des colonnes (doit être identique au X_train)
X_examen_final = df_test_final[features]

# 4. Prédictions
predictions_ultra = mon_modele.predict(X_examen_final)

# 5. Création du fichier CSV final
soumission = pd.DataFrame({
    'Id': df_test.index,
    'temperature': predictions_ultra
})

soumission.to_csv('soumission_xgboost_v1.csv', index=False)
print("Fichier 'soumission_xgboost_v1.csv' généré avec succès ! 🚀")

NameError: name 'modele_final' is not defined

In [37]:
import numpy as np
import pandas as pd

# 1. On récupère le bon modèle peu importe son nom
try:
    mon_modele = modele_final
except NameError:
    try:
        mon_modele = modele_ultra
    except NameError:
        print("❌ ERREUR : Aucun modèle n'a été trouvé en mémoire. "
              "Vérifie que tu as bien lancé la cellule d'entraînement (celle qui prend 1-2 min).")

# 2. Préparation du fichier test
df_test_final = pd.merge(df_test, df_sensors_unique, on='sensor', how='left')
df_test_final['dist_origin'] = np.sqrt(df_test_final['coor_x']**2 + df_test_final['coor_y']**2 + df_test_final['coor_z']**2)

# 3. Sélection et Prédiction
X_examen_final = df_test_final[features]
predictions_ultra = mon_modele.predict(X_examen_final)

# 4. Création du CSV
soumission = pd.DataFrame({
    'Id': df_test.index,
    'temperature': predictions_ultra
})

soumission.to_csv('soumission_xgboost_v1.csv', index=False)
print("✅ Mission accomplie ! Le fichier 'soumission_xgboost_v1.csv' est prêt.")

✅ Mission accomplie ! Le fichier 'soumission_xgboost_v1.csv' est prêt.


In [38]:
# 1. NETTOYAGE ULTIME
df_clean = df_complet.dropna().copy()

# On lisse la température par capteur pour enlever le "bruit"
# Ça permet à l'IA de voir la vraie courbe de chauffe
df_clean['temp_lisse'] = df_clean.groupby('sensor')['temperature'].transform(lambda x: x.rolling(window=5, center=True).mean())
df_clean = df_clean.dropna() # On enlève les nouveaux NaN créés par le lissage

# 2. FEATURE ENGINEERING (Normalisation simple)
# On s'assure que toutes les colonnes sont entre 0 et 1
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
cols_to_scale = ['time', 'power', 'coor_x', 'coor_y', 'coor_z']

df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])

# 3. ENTRAÎNEMENT (On revient à un Random Forest robuste mais plus gros)
from sklearn.ensemble import RandomForestRegressor

X = df_clean[cols_to_scale]
y = df_clean['temp_lisse'] # On apprend sur la version propre !

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modele_robuste = RandomForestRegressor(n_estimators=150, max_depth=15, n_jobs=-1, random_state=42)
modele_robuste.fit(X_train, y_train)

print(f"Erreur locale sur données propres : {mean_absolute_error(y_test, modele_robuste.predict(X_test)):.4f}")

Erreur locale sur données propres : 1.0483


In [39]:
# 1. Préparation du test (Fusion + Distance)
df_test_final = pd.merge(df_test, df_sensors_unique, on='sensor', how='left')

# 2. On applique la même transformation (Scaling) que pour l'entraînement
# TRÈS IMPORTANT : On utilise .transform() et PAS .fit_transform()
X_test_final = df_test_final[cols_to_scale].copy()
X_test_final[cols_to_scale] = scaler.transform(X_test_final[cols_to_scale])

# 3. Prédictions avec le modèle robuste
predictions_podium = modele_robuste.predict(X_test_final)

# 4. Création du CSV
soumission = pd.DataFrame({
    'Id': df_test.index,
    'temperature': predictions_podium
})

soumission.to_csv('soumission_podium_clean.csv', index=False)
print("✅ Fichier 'soumission_podium_clean.csv' prêt !")
print("Ce modèle est lissé et normalisé, il devrait être bien plus stable sur Kaggle.")

✅ Fichier 'soumission_podium_clean.csv' prêt !
Ce modèle est lissé et normalisé, il devrait être bien plus stable sur Kaggle.


In [41]:
import xgboost as xgb

# 1. On reste sur les bases solides de la Soumission 3 (les colonnes qui marchent)
features_v5 = ['time', 'power', 'coor_x', 'coor_y', 'coor_z']
X = df_clean[features_v5]
y = df_clean['temp_lisse']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

# 2. XGBOOST avec "Frein à main" (Early Stopping)
modele_final = xgb.XGBRegressor(
    n_estimators=2000, 
    learning_rate=0.03, # On y va très doucement
    max_depth=6,        # Arbres peu profonds pour éviter l'overfitting
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mae",
    early_stopping_rounds=50 # S'arrête si ça n'évolue plus pendant 50 tours
)

modele_final.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

# 3. Prédiction sur le Test (avec le même scaler que la soumission 3 !)
X_test_final = df_test_final[features_v5].copy()
X_test_final[features_v5] = scaler.transform(X_test_final[features_v5])

preds_v5 = modele_final.predict(X_test_final)

# 4. Génération de soumission_5
soumission_5 = pd.DataFrame({'Id': df_test.index, 'temperature': preds_v5})
soumission_5.to_csv('soumission_5.csv', index=False)
print("Soumission 5 générée. C'est notre meilleure chance technique !")

[0]	validation_0-mae:51.04985
[100]	validation_0-mae:4.85460
[200]	validation_0-mae:2.26357
[300]	validation_0-mae:1.75543
[400]	validation_0-mae:1.47365
[500]	validation_0-mae:1.29216
[600]	validation_0-mae:1.18209
[700]	validation_0-mae:1.10529
[800]	validation_0-mae:1.04730
[900]	validation_0-mae:1.00171
[1000]	validation_0-mae:0.96397
[1100]	validation_0-mae:0.93532
[1200]	validation_0-mae:0.91199
[1300]	validation_0-mae:0.89191
[1400]	validation_0-mae:0.87558
[1500]	validation_0-mae:0.86279
[1600]	validation_0-mae:0.85047
[1700]	validation_0-mae:0.83898
[1800]	validation_0-mae:0.82961
[1900]	validation_0-mae:0.82168
[1999]	validation_0-mae:0.81592
Soumission 5 générée. C'est notre meilleure chance technique !


In [44]:
def extract_top_features(df):
    df = df.copy()
    df = df.sort_values(['sensor', 'time'])
    
    for window in [30, 300]:
        # Calcul sur la PUISSANCE (toujours disponible)
        df[f'power_mean_{window}'] = (
            df.groupby('sensor')['power']
            .transform(lambda x: x.rolling(window=window).mean())
            .bfill()
        )
        
        # Calcul sur la TEMPÉRATURE (uniquement si elle existe !)
        if 'temperature' in df.columns:
            df[f'temp_std_{window}'] = (
                df.groupby('sensor')['temperature']
                .transform(lambda x: x.rolling(window=window).std())
                .fillna(0)
            )
    
    return df

# 1. On prépare le Train (avec temperature)
df_ultra_clean = extract_top_features(df_complet).dropna()

# 2. On entraîne (Note : on n'utilise que les colonnes de puissance pour que ça marche sur le test !)
features_elite = ['time', 'power', 'power_mean_30', 'power_mean_300', 'coor_x', 'coor_y', 'coor_z']
X = df_ultra_clean[features_elite]
y = df_ultra_clean['temperature']

modele_elite = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, n_jobs=-1)
modele_elite.fit(X, y)

# 3. On prépare le Test (SANS temperature, le 'if' dans la fonction va nous sauver)
df_test_elite = pd.merge(df_test, df_sensors_unique, on='sensor', how='left')
df_test_elite = extract_top_features(df_test_elite)

# 4. Prédiction
preds_elite = modele_elite.predict(df_test_elite[features_elite])

# 5. Soumission
soumission_6 = pd.DataFrame({'Id': df_test.index, 'temperature': preds_elite})
soumission_6.to_csv('soumission_6.csv', index=False)
print("Soumission 6 générée proprement ! ✨")

Soumission 6 générée proprement ! ✨
